In [ ]:
pgf_backend = True
# pgf_backend = False
figure_str = "paper_"


import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cycler import cycler

if pgf_backend:
    mpl.use("pgf")

In [ ]:
import sys

sys.path.append("../../src/")


import pickle

from lightning import seed_everything

In [ ]:
# IEEE Access matplotlib settings
# Source: https://journals.ieeeauthorcenter.ieee.org/create-your-ieee-journal-article/create-graphics-for-your-article/resolution-and-size/

font_size = 8

markers = [
    "o",
    "s",
    "^",
    "D",
    "v",
    "P",
    "X",
    "*",
    "<",
    ">",
    "p",
    "h",
    "H",
    "d",
    "|",
    "_",
    "1",
    "2",
    "3",
    "4",
    "+",
    "x",
    ".",
    ",",
]
colors = list(mpl.colormaps["tab10"].colors) * ((len(markers) // 10) + 1)
colors = colors[: len(markers)]

plt.rcParams.update(
    {
        "pgf.rcfonts": False,  # don't use matplotlib defaults
        "pgf.texsystem": "lualatex",  # use LuaLaTeX
        "font.family": "serif",
        "font.serif": ["Times New Roman"],  # system Times New Roman
        "figure.dpi": 300,
        "font.size": font_size,
        "axes.titlesize": font_size,
        "figure.titlesize": font_size,
        "axes.labelsize": font_size,
        "legend.fontsize": font_size,
        "xtick.labelsize": font_size,
        "ytick.labelsize": font_size,
        "pgf.preamble": r"\usepackage{fontspec}\setmainfont{Times New Roman}",
        "axes.prop_cycle": cycler(color=colors)
        + cycler(marker=markers)
        + cycler(markevery=[10] * len(markers)),
        "lines.markersize": 3,
        "lines.markeredgewidth": 0.0,
    }
)

# IEEE standard widths (inches)
linewidth_singlecol = 3.5  # single-column figure
linewidth_doublecol = 7.16  # double-column figure
max_height = 9.25  # max text height

golden_ratio = (5**0.5 - 1) / 2

dpi = 300


def set_size(width=linewidth_singlecol, ratio=golden_ratio, height_pad=0):
    height = width * ratio + height_pad
    return (width, min(height, max_height))

In [ ]:
seed_everything(0)

In [ ]:
models_dict = []

In [ ]:
file_paths = [
    "../evaluate_models/results/metric_evaluation/NaiveHS.pkl",
    "../evaluate_models/results/metric_evaluation/DDNN_Ens.pkl",
    "../evaluate_models/results/metric_evaluation/MCD.pkl",
    "../evaluate_models/results/metric_evaluation/EvDNN.pkl",
    "../evaluate_models/results/metric_evaluation/DDNN_CP.pkl",
    "../evaluate_models/results/metric_evaluation/Ens_CP.pkl",
    "../evaluate_models/results/metric_evaluation/MCD_CP.pkl",
    "../evaluate_models/results/metric_evaluation/EvDNN_CP.pkl",
    "../evaluate_models/results/metric_evaluation/LEAR_GARCH.pkl",
    "../evaluate_models/results/metric_evaluation/LEAR_QRA.pkl",
    "../evaluate_models/results/metric_evaluation/LEAR_CP.pkl",
    "../evaluate_models/results/metric_evaluation/XGBoost_GARCH.pkl",
    "../evaluate_models/results/metric_evaluation/XGBoost_QRA.pkl",
    "../evaluate_models/results/metric_evaluation/XGBoost_CP.pkl",
]

for file_path in file_paths:
    with open(file_path, "rb") as f:
        models_dict.extend(pickle.load(f))

In [ ]:
for model in models_dict:
    print(model["model_name"])

In [ ]:
for model in models_dict:
    if model["model_name"] == "ddnn_normal":
        model["model_name"] = "DDNN"
    elif model["model_name"] == "ens5_normal":
        model["model_name"] = "Ens5"
    elif model["model_name"] == "ens10_normal":
        model["model_name"] = "Ens10"
    elif model["model_name"] == "mcd10_normal":
        model["model_name"] = "MCD10"
    elif model["model_name"] == "mcd30_normal":
        model["model_name"] = "MCD30"
    elif model["model_name"] == "naive_hs_train_normal":
        model["model_name"] = "Naive-HS$_{train}$"
    elif model["model_name"] == "naive_hs_val_normal":
        model["model_name"] = "Naive-HS$_{val}$"
    elif model["model_name"] == "evdnn_normal":
        model["model_name"] = "EvDNN"
    elif model["model_name"] == "lasso_garch":
        model["model_name"] = "LASSO-GARCH"
    elif model["model_name"] == "LEAR_GARCH":
        model["model_name"] = "LEAR-GARCH"
    elif model["model_name"] == "LEAR_QRA":
        model["model_name"] = "LEAR-QRA"
    elif model["model_name"] == "XGBoost_GARCH":
        model["model_name"] = "XGBoost-GARCH"
    elif model["model_name"] == "XGBoost_QRA":
        model["model_name"] = "XGBoost-QRA"

    if "ece" in model.keys():
        model["maace"] = model["ece"]
        model["maace_std"] = model["ece_std"]

    model["maace"] = model["maace"] * 100
    model["maace_std"] = model["maace_std"] * 100
    model["MAE"] = model["mae"]
    model["MAE std"] = model["mae_std"]
    model["RMSE"] = model["rmse"]
    model["RMSE std"] = model["rmse_std"]
    model["CRPS$_{1:99:1}$"] = model["crps"]
    model["CRPS$_{1:99:1}$ std"] = model["crps_std"]
    model["MAACE$_{2:98:2}$"] = model["maace"]
    model["MAACE$_{2:98:2}$ std"] = model["maace_std"]

In [ ]:
metrics = [
    "MAE",
    "MAE std",
    "RMSE",
    "RMSE std",
    "CRPS$_{1:99:1}$",
    "CRPS$_{1:99:1}$ std",
    "MAACE$_{2:98:2}$",
    "MAACE$_{2:98:2}$ std",
]
results = {
    metric: [np.mean(model[metric]) for model in models_dict] for metric in metrics
}

model_names = [model["model_name"] for model in models_dict]
results_df = pd.DataFrame(results, index=model_names)
results_df.round(3)

In [ ]:
print(results_df.round(3).to_latex(float_format="%.3f"))

In [ ]:
quantiles = np.linspace(0.01, 0.99, 99)
confidence_levels = np.flip(
    np.array([quantiles[-i - 1] - quantiles[i] for i in range(len(quantiles) // 2)])
)
confidence_levels

In [ ]:
eval_confidence_levels_indx = [24, 29, 34, 39, 44, 48]
print(confidence_levels[eval_confidence_levels_indx])

In [ ]:
results = {
    f"PICP$_{int(confidence_levels[cl_indx]*100)}%$": [
        model["coverage_mean"][cl_indx] * 100 for model in models_dict
    ]
    for cl_indx in eval_confidence_levels_indx
}
# results = {
#     f"PICP$_{int(confidence_levels[cl_indx]*100)}%$": [
#         model["coverage_std"][cl_indx] * 100 for model in models_dict
#     ]
#     for cl_indx in eval_confidence_levels_indx
# }

model_names = [model["model_name"] for model in models_dict]
results_df = pd.DataFrame(results, index=model_names)
results_df.round(2)

In [ ]:
print(results_df.round(2).to_latex(float_format="%.2f"))

In [ ]:
results = {
    f"MPIW$_{int(confidence_levels[cl_indx]*100)}%$": [
        model["mpiw_mean"][cl_indx] for model in models_dict
    ]
    for cl_indx in eval_confidence_levels_indx
}
# results = {
#     f"MPIW$_{int(confidence_levels[cl_indx]*100)}%$": [
#         model["mpiw_std"][cl_indx] for model in models_dict
#     ]
#     for cl_indx in eval_confidence_levels_indx
# }

model_names = [model["model_name"] for model in models_dict]
results_df = pd.DataFrame(results, index=model_names)
results_df.round(2)

In [ ]:
print(results_df.round(2).to_latex(float_format="%.2f"))

In [ ]:
models_to_exclude = {
    "Ens5",
    "MCD10",
    "Naive-HS$_{train}$",
    "EvDNN",
    "EvDNN-CP",
    "MCD30-CP",
    "Ens10-CP",
}
models_dict = [
    model for model in models_dict if model["model_name"] not in models_to_exclude
]

In [ ]:

# plot coverage probability
# figsize = set_size(height_pad=1.5)
figsize = set_size(width=linewidth_doublecol*0.8)#, height_pad=1.5)
fig, ax = plt.subplots(
    2, 1, figsize=figsize, sharex=True, gridspec_kw={"height_ratios": [1, 1]}
)
lw = 1
ax[0].plot(
    confidence_levels * 100,
    confidence_levels * 100,
    linestyle="--",
    lw=lw,
    color="gray",
    label="Optimal",
    marker="",
)
for i, model in enumerate(models_dict):
    ax[0].plot(
        confidence_levels * 100,
        model["coverage_mean"] * 100,
        label=model["model_name"],
        alpha=0.8,
        lw=lw,
        color=f"C{i}",
        marker=markers[i],
    )
    # ax[0].scatter(
    #     confidence_levels * 100,
    #     model["coverage_mean"] * 100,
    #     color=f"C{i}",
    #     s=5,
    #     alpha=0.8,
    #     marker="",
    #     rasterized=True,
    # )
# # plot std deviation
# for model in models_dict:
#     ax.fill_between(
#         confidence_levels * 100,
#         (model["coverage_mean"] - model["coverage_std"]) * 100,
#         (model["coverage_mean"] + model["coverage_std"]) * 100,
#         alpha=0.2,
#     )

ax[0].set_xlim(0, 100)
ax[0].set_ylim(0, 100)
ax[0].grid(alpha=0.4)
ax[1].set_xlabel("Confidence level [%]")
ax[0].set_ylabel("PICP [%]")
# ax.legend()
ax[0].legend(
    loc="lower left",
    ncol=4,
    bbox_to_anchor=(0, 1),
)


# # add second y-axis
# ax2 = ax[0].twinx()
# # plot std
# ax2.set_ylim(0, 20)
# ax2.set_ylabel("std [%]")
# ax2.grid(False)

# for model in models_dict:
#     ax2.plot(
#         confidence_levels * 100,
#         model["coverage_std"] * 100,
#         label=model["model_name"],
#         alpha=0.8,
#         linestyle=":",
#         lw=lw,
#     )

model_ddnn = [model for model in models_dict if model["model_name"] == "DDNN"][0]

# plot difference of ddnn to optimal
ax[1].plot(
    confidence_levels * 100,
    (confidence_levels - model_ddnn["coverage_mean"]) * 100,
    linestyle="--",
    lw=lw,
    color="gray",
    label="Optimal",
    marker="",
)

for i, model in enumerate(models_dict):
    ax[1].plot(
        confidence_levels * 100,
        (model["coverage_mean"] - model_ddnn["coverage_mean"]) * 100,
        label=model["model_name"],
        alpha=0.8,
        lw=lw,
        color=f"C{i}",
        marker=markers[i],
    )
ax[1].set_xlim(0, 100)
ax[1].set_ylim(-5, 15)
ax[1].set_ylabel("Diff with DDNN [%]")
ax[1].grid(alpha=0.4)


plt.tight_layout()
if pgf_backend:
    plt.savefig(
        "plots/" + figure_str + "results_point_interval_picp_and_diff.pdf",
        bbox_inches="tight",
        dpi=dpi,
    )

# --- Graphical Abstract (660x295 px JPG, <45 KB) ---
import io
import os
from PIL import Image
from matplotlib.backends.backend_agg import FigureCanvasAgg

canvas = FigureCanvasAgg(fig)
canvas.draw()
buf = io.BytesIO()
canvas.print_png(buf)
buf.seek(0)
img = Image.open(buf).convert("RGB")

# Scale to fit within 660x295 preserving aspect ratio, pad remainder with white
img.thumbnail((660, 295), Image.LANCZOS)
ga_canvas = Image.new("RGB", (660, 295), (255, 255, 255))
offset = ((660 - img.width) // 2, (295 - img.height) // 2)
ga_canvas.paste(img, offset)

ga_path = "plots/" + figure_str + "graphical_abstract.jpg"
ga_canvas.save(ga_path, format="JPEG", quality=95, optimize=True)
print(f"Graphical abstract saved: {os.path.getsize(ga_path) / 1024:.1f} KB  ({img.width}x{img.height} → 660x295)")
# ---

    
# plt.show()



In [ ]:
# plot coverage probability
# figsize = set_size(height_pad=1.5)
figsize = set_size(width=linewidth_doublecol*0.8)#, height_pad=1.5)
fig, ax = plt.subplots(
    2, 1, figsize=figsize, sharex=True, gridspec_kw={"height_ratios": [1, 1]}
)
lw = 1
ax[0].plot(
    confidence_levels * 100,
    confidence_levels * 100,
    linestyle="--",
    lw=lw,
    color="gray",
    label="Optimal",
    marker="",
)
for i, model in enumerate(models_dict):
    ax[0].plot(
        confidence_levels * 100,
        model["coverage_mean"] * 100,
        label=model["model_name"],
        alpha=0.8,
        lw=lw,
        color=f"C{i}",
        marker=markers[i],
    )
    # ax[0].scatter(
    #     confidence_levels * 100,
    #     model["coverage_mean"] * 100,
    #     color=f"C{i}",
    #     s=5,
    #     alpha=0.8,
    #     marker="",
    #     rasterized=True,
    # )
# # plot std deviation
# for model in models_dict:
#     ax.fill_between(
#         confidence_levels * 100,
#         (model["coverage_mean"] - model["coverage_std"]) * 100,
#         (model["coverage_mean"] + model["coverage_std"]) * 100,
#         alpha=0.2,
#     )

ax[0].set_xlim(0, 100)
ax[0].set_ylim(0, 100)
ax[0].grid(alpha=0.4)
ax[1].set_xlabel("Confidence level [%]")
ax[0].set_ylabel("PICP [%]")
# ax.legend()
ax[0].legend(
    loc="lower left",
    ncol=4,
    bbox_to_anchor=(0, 1),
)


# # add second y-axis
# ax2 = ax[0].twinx()
# # plot std
# ax2.set_ylim(0, 20)
# ax2.set_ylabel("std [%]")
# ax2.grid(False)

# for model in models_dict:
#     ax2.plot(
#         confidence_levels * 100,
#         model["coverage_std"] * 100,
#         label=model["model_name"],
#         alpha=0.8,
#         linestyle=":",
#         lw=lw,
#     )

model_ddnn = [model for model in models_dict if model["model_name"] == "DDNN"][0]

# plot difference of ddnn to optimal
ax[1].plot(
    confidence_levels * 100,
    (confidence_levels - model_ddnn["coverage_mean"]) * 100,
    linestyle="--",
    lw=lw,
    color="gray",
    label="Optimal",
    marker="",
)

for i, model in enumerate(models_dict):
    ax[1].plot(
        confidence_levels * 100,
        (model["coverage_mean"] - model_ddnn["coverage_mean"]) * 100,
        label=model["model_name"],
        alpha=0.8,
        lw=lw,
        color=f"C{i}",
        marker=markers[i],
    )
ax[1].set_xlim(0, 100)
ax[1].set_ylim(-5, 15)
ax[1].set_ylabel("Diff with DDNN [%]")
ax[1].grid(alpha=0.4)


plt.tight_layout()
if pgf_backend:
    plt.savefig(
        "plots/" + figure_str + "results_point_interval_picp_and_diff.pdf",
        bbox_inches="tight",
        dpi=dpi,
    )
    
    
# plt.show()

In [ ]:
# plot mean prediction interval width
# figsize = set_size(height_pad=1.5)
figsize = set_size(width=linewidth_doublecol*0.8)#, height_pad=1.5)
fig, ax = plt.subplots(
    2, 1, figsize=figsize, sharex=True, gridspec_kw={"height_ratios": [1, 1]}
)

lw = 1
for model in models_dict:
    ax[0].plot(
        confidence_levels * 100,
        model["mpiw_mean"],
        label=model["model_name"],
        alpha=0.8,
        lw=lw,
    )

ax[0].set_xlim(0, 100)
ax[0].set_ylim(0, 200)
ax[0].grid(alpha=0.4)
ax[1].set_xlabel("Confidence level [%]")
ax[0].set_ylabel("MPIW [EUR]")
ax[0].legend(ncols=3)
# ax[0].legend(
#     loc="lower left",
#     ncol=2,
#     bbox_to_anchor=(0, 1),
# )

# # add second y-axis
# ax2 = ax[0].twinx()
# # plot std
# ax2.set_ylim(0, 40)
# ax2.set_ylabel("std [EUR]")
# ax2.grid(False)
# for model in models_dict:
#     ax2.plot(
#         confidence_levels * 100,
#         model["mpiw_std"],
#         label=model["model_name"],
#         alpha=0.8,
#         linestyle=":",
#         lw=lw,
#     )

model_ddnn = [model for model in models_dict if model["model_name"] == "DDNN"][0]
for i, model in enumerate(models_dict):
    ax[1].plot(
        confidence_levels * 100,
        (model["mpiw_mean"] - model_ddnn["mpiw_mean"]),
        label=model["model_name"],
        alpha=0.8,
        lw=lw,
        color=f"C{i}",
        marker=markers[i],
    )
ax[1].set_xlim(0, 100)
ax[1].set_ylim(-10, 20)
ax[1].set_ylabel("Diff with DDNN [EUR]")
ax[1].grid(alpha=0.4)

plt.tight_layout()
if pgf_backend:
    plt.savefig(
        "plots/" + figure_str + "results_point_interval_mpiw_and_diff.pdf",
        bbox_inches="tight",
        dpi=dpi,
    )
# plt.show()

In [ ]:
# plot mean prediction interval width
# figsize = set_size(height_pad=1.5)
figsize = set_size(width=linewidth_doublecol*0.8)#, height_pad=1.5)
fig, ax = plt.subplots(
    2, 1, figsize=figsize, sharex=True, gridspec_kw={"height_ratios": [1, 1]}
)

lw = 1
for i, model in enumerate(models_dict):
    ax[0].plot(
        model["coverage_mean"] * 100,
        model["mpiw_mean"],
        label=model["model_name"],
        marker="",
        color=f"C{i}",
        # markersize=2,
        alpha=0.8,
        lw=lw,
    )
    ax[0].scatter(
        model["coverage_mean"] * 100,
        model["mpiw_mean"],
        color=f"C{i}",
        s=5,
        alpha=0.8,
        marker=markers[i],
        rasterized=True,
    )
        
        
        
ax[0].set_xlim(0, 100)
# ax.set_ylim(0, 200)
ax[0].grid(alpha=0.4)
ax[1].set_xlabel("PICP [%]")
ax[0].set_ylabel("MPIW [EUR]")
ax[0].legend(ncols=3)
# ax[0].legend(
#     loc="lower left",
#     ncol=2,
#     bbox_to_anchor=(0, 1),
# )

# # add second y-axis
# ax2 = ax[0].twinx()
# # plot std
# ax2.set_ylim(0, 40)
# ax2.set_ylabel("std [EUR]")
# ax2.grid(False)
# for model in models_dict:
#     ax2.plot(
#         model["coverage_mean"] * 100,
#         model["mpiw_std"],
#         label=model["model_name"],
#         alpha=0.8,
#         linestyle=":",
#         lw=lw,
#     )

model_ddnn = [model for model in models_dict if model["model_name"] == "DDNN"][0]
for i, model in enumerate(models_dict):
    picp_lbound = np.max(
        [np.min(model["coverage_mean"]), np.min(model_ddnn["coverage_mean"])]
    )
    picp_ubound = np.min(
        [np.max(model["coverage_mean"]), np.max(model_ddnn["coverage_mean"])]
    )

    # interpolate the values
    picp_interp = np.linspace(picp_lbound, picp_ubound, 100)
    model_interp = np.interp(picp_interp, model["coverage_mean"], model["mpiw_mean"])
    model_ddnn_interp = np.interp(
        picp_interp, model_ddnn["coverage_mean"], model_ddnn["mpiw_mean"]
    )

    ax[1].plot(
        picp_interp * 100,
        (model_interp - model_ddnn_interp),
        label=model["model_name"],
        marker=markers[i],
        color=f"C{i}",
        # markersize=2,
        alpha=0.8,
        lw=lw,
    )
    # ax[1].scatter(
    #     picp_interp * 100,
    #     (model_interp - model_ddnn_interp),
    #     color=f"C{i}",
    #     s=5,
    #     alpha=0.8,
    #     marker=markers[i],
    #     rasterized=True,
    # )
    
ax[1].set_xlim(0, 100)
ax[1].set_ylim(-30, 5)
ax[1].set_ylabel("Diff with DDNN [EUR]")
ax[1].grid(alpha=0.4)

plt.tight_layout()
if pgf_backend:
    plt.savefig(
        "plots/" + figure_str + "results_point_interval_picp_mpiw_and_diff.pdf",
        bbox_inches="tight",
        dpi=dpi,
    )
# plt.show()